In [7]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

In [9]:
from __future__ import annotations

import json
import logging
from pathlib import Path

import pandas as pd

from data.features import build_panel
from data.vocab import Vocabulary

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger(__name__)

In [2]:
from pathlib import Path

Path.cwd()

WindowsPath('c:/stocktwits_2026/StockTwit_WM/notebooks')

In [42]:
RAW_DIR = Path(r"C:\stocktwits_2026\parquet\features_wo_messages")
OUT_DIR = Path(r"C:\stocktwits_2026\StockTwit_WM\data\processed_week")

TOP_K = 200
MIN_WEEKS = 10
PARQUET_GLOB = "**/*.parquet"

OUT_DIR.mkdir(parents=True, exist_ok=True)

In [43]:
from pathlib import Path

RAW_DIR = Path(r"C:\stocktwits_2026\parquet\feature_wo_messages")

files = list(RAW_DIR.rglob("*.parquet"))
print("Number of parquet files:", len(files))

for f in files[:10]:
    print(f)

Number of parquet files: 1234
C:\stocktwits_2026\parquet\feature_wo_messages\year=2008\month=10\part.0.parquet
C:\stocktwits_2026\parquet\feature_wo_messages\year=2008\month=10\part.10.parquet
C:\stocktwits_2026\parquet\feature_wo_messages\year=2008\month=10\part.15.parquet
C:\stocktwits_2026\parquet\feature_wo_messages\year=2008\month=10\part.20.parquet
C:\stocktwits_2026\parquet\feature_wo_messages\year=2008\month=10\part.242.parquet
C:\stocktwits_2026\parquet\feature_wo_messages\year=2008\month=10\part.25.parquet
C:\stocktwits_2026\parquet\feature_wo_messages\year=2008\month=10\part.30.parquet
C:\stocktwits_2026\parquet\feature_wo_messages\year=2008\month=10\part.36.parquet
C:\stocktwits_2026\parquet\feature_wo_messages\year=2008\month=10\part.5.parquet
C:\stocktwits_2026\parquet\feature_wo_messages\year=2008\month=11\part.36.parquet


In [44]:
panel = build_panel(
    parquet_dir=RAW_DIR,
    output_path=OUT_DIR / "panel_all.parquet",
    top_k=TOP_K,
)

print("Panel shape:", panel.shape)
print("Weeks:", panel["week"].nunique())
print("Unique tickers:", panel["symbol"].nunique())

panel.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[features] raw panel: 4,210,119 rows, 26,016 symbols, 762 weeks
[features] panel saved: 151,337 rows → C:\stocktwits_2026\StockTwit_WM\data\processed_week\panel_all.parquet
Panel shape: (151337, 11)
Weeks: 762
Unique tickers: 5664


,symbol,week,msg_count,user_count,bullish_count,labeled_count,log_attention,bullish_rate,bearish_rate,unlabeled_rate,attn_growth
0,AAPL,2008-05-26,8,6,0.0,0.0,2.197225,0.0,1.0,1.0,0.0
1,ACI,2008-05-26,1,1,0.0,0.0,0.693147,0.0,1.0,1.0,0.0
2,AMAT,2008-05-26,1,1,0.0,0.0,0.693147,0.0,1.0,1.0,0.0
3,BB,2008-05-26,3,3,0.0,0.0,1.386294,0.0,1.0,1.0,0.0
4,BGPIQ,2008-05-26,1,1,0.0,0.0,0.693147,0.0,1.0,1.0,0.0


In [45]:
print(panel.columns.tolist())
print(panel.dtypes)

panel.describe(include="all")

['symbol', 'week', 'msg_count', 'user_count', 'bullish_count', 'labeled_count', 'log_attention', 'bullish_rate', 'bearish_rate', 'unlabeled_rate', 'attn_growth']
symbol                       str
week              datetime64[us]
msg_count                  int64
user_count                 int64
bullish_count            float64
labeled_count            float64
log_attention            float64
bullish_rate             float64
bearish_rate             float64
unlabeled_rate           float64
attn_growth              float64
dtype: object


,symbol,week,msg_count,user_count,bullish_count,labeled_count,log_attention,bullish_rate,bearish_rate,unlabeled_rate,attn_growth
count,151337,151337,151337.000000,151337.000000,151337.000000,151337.000000,151337.000000,151337.000000,151337.000000,151337.000000,151337.000000
unique,5664,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,AAPL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,762,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,2015-09-28 22:05:23.944574,1304.986256,296.229243,530.898029,627.744815,5.795190,0.632729,0.367271,0.735601,0.031045
min,NaN,2008-05-26 00:00:00,1.000000,1.000000,0.000000,0.000000,0.693147,0.000000,0.000000,0.011289,-0.737350
25%,NaN,2012-02-13 00:00:00,113.000000,42.000000,2.000000,2.000000,4.736198,0.380697,0.058974,0.539941,-0.053067
50%,NaN,2015-09-28 00:00:00,382.000000,123.000000,68.000000,94.000000,5.948035,0.796552,0.203448,0.752483,0.000000
75%,NaN,2019-05-13 00:00:00,1055.000000,296.000000,358.000000,434.000000,6.962243,0.941026,0.619303,0.981132,0.064474
max,NaN,2022-12-26 00:00:00,447101.000000,54252.000000,281766.000000,299581.000000,13.010542,1.000000,1.000000,1.000000,5.000000


In [46]:
panel["week"] = pd.to_datetime(panel["week"])

train_mask = panel["week"] < "2019-01-01"
val_mask   = (panel["week"] >= "2019-01-01") & (panel["week"] < "2020-01-01")
test1_mask = (panel["week"] >= "2020-01-01") & (panel["week"] < "2020-07-01")
test2_mask = (panel["week"] >= "2020-10-01") & (panel["week"] < "2021-07-01")

panel_train = panel[train_mask].copy()
panel_val   = panel[val_mask].copy()
panel_test1 = panel[test1_mask].copy()
panel_test2 = panel[test2_mask].copy()

print("Train:", panel_train.shape)
print("Val:", panel_val.shape)
print("Test1 COVID:", panel_test1.shape)
print("Test2 GME:", panel_test2.shape)

Train: (109737, 11)
Val: (10400, 11)
Test1 COVID: (5200, 11)
Test2 GME: (7800, 11)


In [47]:
ticker_counts = panel_train.groupby("symbol")["week"].nunique()
eligible = ticker_counts[ticker_counts >= MIN_WEEKS].index.tolist()

print("Eligible tickers:", len(eligible))

vocab = Vocabulary.build(eligible)
vocab.save(OUT_DIR / "vocab.json")

print("Vocabulary size including PAD:", len(vocab))

Eligible tickers: 1520
[vocab] built: 1520 tickers (indices 1–1520)
[vocab] saved 1520 tickers → C:\stocktwits_2026\StockTwit_WM\data\processed_week\vocab.json
Vocabulary size including PAD: 1520


In [48]:
panel.to_parquet(OUT_DIR / "panel_all.parquet", index=False)
panel_train.to_parquet(OUT_DIR / "panel_train.parquet", index=False)
panel_val.to_parquet(OUT_DIR / "panel_val.parquet", index=False)
panel_test1.to_parquet(OUT_DIR / "panel_test1.parquet", index=False)
panel_test2.to_parquet(OUT_DIR / "panel_test2.parquet", index=False)

print("Saved processed parquet files to:", OUT_DIR)

Saved processed parquet files to: C:\stocktwits_2026\StockTwit_WM\data\processed_week


In [49]:
stats = {
    "top_k": TOP_K,
    "min_weeks": MIN_WEEKS,
    "vocab_size": len(vocab),
    "n_train_rows": len(panel_train),
    "n_val_rows": len(panel_val),
    "n_test1_rows": len(panel_test1),
    "n_test2_rows": len(panel_test2),
    "train_weeks": panel_train["week"].nunique(),
    "val_weeks": panel_val["week"].nunique(),
    "test1_weeks": panel_test1["week"].nunique(),
    "test2_weeks": panel_test2["week"].nunique(),
    "feature_cols": [
        "log_attention",
        "bullish_rate",
        "bearish_rate",
        "unlabeled_rate",
        "attn_growth",
    ],
}

with open(OUT_DIR / "dataset_stats.json", "w") as f:
    json.dump(stats, f, indent=2)

stats

{'top_k': 200,
 'min_weeks': 10,
 'vocab_size': 1520,
 'n_train_rows': 109737,
 'n_val_rows': 10400,
 'n_test1_rows': 5200,
 'n_test2_rows': 7800,
 'train_weeks': 554,
 'val_weeks': 52,
 'test1_weeks': 26,
 'test2_weeks': 39,
 'feature_cols': ['log_attention',
  'bullish_rate',
  'bearish_rate',
  'unlabeled_rate',
  'attn_growth']}